### Loading saved model (ELU activation function) --> quantization didnt work

In [ ]:
from tensorflow import keras
model_path = "/Users/noshin/Desktop/AUS/Masters/Research/New_dataset/saved_models/model_freq_101.h5"
model = keras.models.load_model(model_path1)


In [ ]:
model.summary()

Model: "sequential_28"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_48 (Conv2D)          (None, 99, 2, 32)         128       
                                                                 
 batch_normalization_54 (Ba  (None, 99, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_69 (ELU)                (None, 99, 2, 32)         0         
                                                                 
 conv2d_49 (Conv2D)          (None, 97, 2, 32)         3104      
                                                                 
 batch_normalization_55 (Ba  (None, 97, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_70 (ELU)                (None, 97, 2, 32)       

In [ ]:
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow import keras

quantize_model = tfmot.quantization.keras.quantize_model
qat_model = quantize_model(model) 


RuntimeError: Layer elu_69:<class 'tf_keras.src.layers.activation.elu.ELU'> is not supported. You can quantize this layer by passing a `tfmot.quantization.keras.QuantizeConfig` instance to the `quantize_annotate_layer` API.

### Model with ReLU activation

In [107]:
# Reproducibility Settings
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [108]:
import scipy.io
import random
import numpy as np

In [109]:
# Load the .mat file
CTF_class_mov = scipy.io.loadmat('Datasets/CTF_Class_mov_final.mat')
CTF_class_static = scipy.io.loadmat('Datasets/CTF_Class_static_final.mat')

CTF_corridor_mov = scipy.io.loadmat('Datasets/CTF_Corridor_mov_final.mat')
CTF_corridor_static = scipy.io.loadmat('Datasets/CTF_Corridor_static_final.mat')

CTF_lab_mov = scipy.io.loadmat('Datasets/CTF_Lab_mov_final.mat')
CTF_lab_static = scipy.io.loadmat('Datasets/CTF_Lab_static_final.mat')

CTF_SC_mov = scipy.io.loadmat('Datasets/CTF_SC_mov_final.mat')
CTF_SC_static = scipy.io.loadmat('Datasets/CTF_SC_static_final.mat')

In [110]:
# Access the 4th item in each loaded variable
#static
CTF_class_static_array = CTF_class_static[list(CTF_class_static.keys())[3]].T
CTF_corridor_static_array = CTF_corridor_static[list(CTF_corridor_static.keys())[3]].T
CTF_lab_static_array = CTF_lab_static[list(CTF_lab_static.keys())[3]].T
CTF_SC_static_array = CTF_SC_static[list(CTF_SC_static.keys())[3]].T

# mov
CTF_class_mov_array = CTF_class_mov[list(CTF_class_mov.keys())[3]].T
CTF_corridor_mov_array = CTF_corridor_mov[list(CTF_corridor_mov.keys())[3]].T
CTF_lab_mov_array = CTF_lab_mov[list(CTF_lab_mov.keys())[3]].T
CTF_SC_mov_array = CTF_SC_mov[list(CTF_SC_mov.keys())[3]].T

CTF_class_static_array.shape

(38800, 2, 101)

In [111]:
# Combine real and imaginary parts for the 4th item in each loaded variable using stack
CTF_class = np.concatenate((CTF_class_static_array, CTF_class_mov_array), axis=0)
CTF_corridor = np.concatenate((CTF_corridor_static_array, CTF_corridor_mov_array), axis=0)
CTF_lab = np.concatenate((CTF_lab_static_array, CTF_lab_mov_array), axis=0)
CTF_SC = np.concatenate((CTF_SC_static_array, CTF_SC_mov_array), axis=0)

In [112]:
CTF_SC.shape

(77600, 2, 101)

In [113]:
# Combine 145 training points randomly selected from each env dataset into a single training set array, 
# and combine 49 testing points randomly selected from each dataset into a single testing set array.

# Number of unique grid points
num_grid_points = 194*2 #388

# Number of measurements per grid point
num_measurements = 200

# Number of grid points to select for training
num_train_points = int(0.75 * num_grid_points)  # 291

# Initialize empty lists for training and test sets
train_set = []
test_set = []

# For each array (representing a different environment)
for array in [CTF_class, CTF_corridor, CTF_lab, CTF_SC]:
    # Reshape the array to separate the measurements for each grid point
    reshaped_array = array.reshape(num_grid_points, num_measurements, -1, 2)
    
    # Randomly select grid points for training
    train_points = random.sample(range(num_grid_points), num_train_points)
    #print(len(train_points))
        
    # Get boolean array for test points
    test_points_bool = ~np.isin(range(num_grid_points), train_points)

    # Print test points
    # test_points = np.arange(num_grid_points)[test_points_bool]
    # print("Test points len:", len(test_points.tolist()))
    
    # Add the selected grid points to the training set and the rest to the test set
    train_set.append(reshaped_array[train_points])
    test_set.append(reshaped_array[test_points_bool])
    
    # ~ is the logical NOT operator, so ~np.isin(shuffle_index, train_points) gives all the indices not in train_points.
# Concatenate the data from all environments
train_set = np.concatenate(train_set, axis=0)
test_set = np.concatenate(test_set, axis=0)

print("Training set:",train_set.shape)
print("Testing set:",test_set.shape)

Training set: (1164, 200, 101, 2)
Testing set: (388, 200, 101, 2)


In [114]:
# Reshaping the arrays
large_X_train = train_set.reshape([-1,101,2])
large_X_test = test_set.reshape([-1,101,2])
print("large_X_train after reshape:", large_X_train.shape)
print("large_X_test after reshape:", large_X_test.shape)

large_X_train after reshape: (232800, 101, 2)
large_X_test after reshape: (77600, 101, 2)


In [115]:
# Create Labels for training data
#static
label1 = np.ones([291*200]) * 0  # class
label2 = np.ones([291*200]) * 1  # corridor
label3 = np.ones([291*200]) * 2  # lab
label4 = np.ones([291*200]) * 3  # SC

large_Y_train = np.concatenate([label1, label2, label3, label4])

label5 = np.ones([97*200]) * 0  # class
label6 = np.ones([97*200]) * 1  # corridor
label7 = np.ones([97*200]) * 2  # lab
label8 = np.ones([97*200]) * 3  # SC

large_Y_test = np.concatenate([label5, label6, label7, label8])

print("Training labels:", large_Y_train.shape)
print("Testing labels:", large_Y_test.shape)

Training labels: (232800,)
Testing labels: (77600,)


In [116]:
# Shuffle Data
shuffle_index1 = random.sample(range(0,232800), 232800)
#print(shuffle_index1)

# Shuffle data using the shuffled indices
large_X_train_new = large_X_train[shuffle_index1, :, :]
print(shuffle_index1[232797:])

large_Y_train = large_Y_train[shuffle_index1]
large_Y_train

[221954, 135391, 133055]


array([1., 2., 3., ..., 3., 2., 2.])

In [117]:
# Shuffle testing data
shuffle_index2 = random.sample(range(0,77600), 77600)

# Shuffle testing data using the shuffled indices
large_X_test_new = large_X_test[shuffle_index2, :, :]
#print(shuffle_index2[0:3])
print(shuffle_index2[77597:])

large_Y_test = large_Y_test[shuffle_index2]
large_Y_test

[58764, 14174, 70883]


array([2., 2., 2., ..., 3., 0., 3.])

In [118]:
from keras.utils import to_categorical

# Converting labels to categorical one-hot encoding
y_train = to_categorical(large_Y_train, num_classes=4) # (232000,)
y_test = to_categorical(large_Y_test, num_classes=4) # (78400,)

X_train = large_X_train_new # (232000, 101, 2) 
X_test = large_X_test_new # (77600, 101, 2) 

In [119]:
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training labels: (232800, 4)
Testing labels: (77600, 4)
Training data: (232800, 101, 2)
Testing data: (77600, 101, 2)


In [133]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, BatchNormalization, ReLU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler

# Build model
model = Sequential([
    Conv2D(5, kernel_size=(3, 3), input_shape=(101, 2, 1), strides=(1,1), padding="same"), 
    BatchNormalization(),
    ReLU(),

    Conv2D(5, kernel_size=(3, 3), padding="same"),
    BatchNormalization(),
    ReLU(),
    
    Flatten(),
    Dense(64),
    ReLU(),
    Dropout(0.4),
    Dense(4, activation='softmax')
])

def lr_schedule(epoch):
    lr = 0.001
    if epoch > 2:
        lr *= 0.5e-3
    elif epoch > 4:
        lr *= 1e-3
    return lr

lr_scheduler = LearningRateScheduler(lr_schedule)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule(0))

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])


In [134]:
model.summary()

Model: "sequential_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_33 (Conv2D)          (None, 101, 2, 5)         50        
                                                                 
 batch_normalization_31 (Ba  (None, 101, 2, 5)         20        
 tchNormalization)                                               
                                                                 
 re_lu_46 (ReLU)             (None, 101, 2, 5)         0         
                                                                 
 conv2d_34 (Conv2D)          (None, 101, 2, 5)         230       
                                                                 
 batch_normalization_32 (Ba  (None, 101, 2, 5)         20        
 tchNormalization)                                               
                                                                 
 re_lu_47 (ReLU)             (None, 101, 2, 5)       

In [135]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=256, callbacks=[lr_scheduler])

Epoch 1/10
910/910 [==============================] - 40s 41ms/step - loss: 0.1195 - accuracy: 0.9599 - val_loss: 90.1525 - val_accuracy: 0.2500 - lr: 0.0010
Epoch 2/10
910/910 [==============================] - 37s 41ms/step - loss: 0.0108 - accuracy: 0.9972 - val_loss: 37.4695 - val_accuracy: 0.2606 - lr: 0.0010
Epoch 3/10
910/910 [==============================] - 36s 39ms/step - loss: 0.0057 - accuracy: 0.9985 - val_loss: 725.2052 - val_accuracy: 0.2500 - lr: 0.0010
Epoch 4/10
910/910 [==============================] - 37s 41ms/step - loss: 0.0051 - accuracy: 0.9985 - val_loss: 0.0714 - val_accuracy: 0.9793 - lr: 5.0000e-07
Epoch 5/10
910/910 [==============================] - 38s 42ms/step - loss: 0.0046 - accuracy: 0.9987 - val_loss: 0.0697 - val_accuracy: 0.9797 - lr: 5.0000e-07
Epoch 6/10
910/910 [==============================] - 47s 51ms/step - loss: 0.0045 - accuracy: 0.9987 - val_loss: 0.0693 - val_accuracy: 0.9798 - lr: 5.0000e-07
Epoch 7/10
910/910 [======================

In [80]:
# Access the 4th item in each loaded variable
#static
CTF_class_static_array = CTF_class_static[list(CTF_class_static.keys())[3]].T
CTF_corridor_static_array = CTF_corridor_static[list(CTF_corridor_static.keys())[3]].T
CTF_lab_static_array = CTF_lab_static[list(CTF_lab_static.keys())[3]].T
CTF_SC_static_array = CTF_SC_static[list(CTF_SC_static.keys())[3]].T

# mov
CTF_class_mov_array = CTF_class_mov[list(CTF_class_mov.keys())[3]].T
CTF_corridor_mov_array = CTF_corridor_mov[list(CTF_corridor_mov.keys())[3]].T
CTF_lab_mov_array = CTF_lab_mov[list(CTF_lab_mov.keys())[3]].T
CTF_SC_mov_array = CTF_SC_mov[list(CTF_SC_mov.keys())[3]].T

CTF_class_static_array.shape

(38800, 2, 101)

In [ ]:
# Combine real and imaginary parts for the 4th item in each loaded variable using stack
CTF_class = np.concatenate((CTF_class_static_array, CTF_class_mov_array), axis=0)
CTF_corridor = np.concatenate((CTF_corridor_static_array, CTF_corridor_mov_array), axis=0)
CTF_lab = np.concatenate((CTF_lab_static_array, CTF_lab_mov_array), axis=0)
CTF_SC = np.concatenate((CTF_SC_static_array, CTF_SC_mov_array), axis=0)

In [ ]:
CTF_SC.shape

(77600, 2, 101)

In [ ]:
# Combine 145 training points randomly selected from each env dataset into a single training set array, 
# and combine 49 testing points randomly selected from each dataset into a single testing set array.

# Number of unique grid points
num_grid_points = 194*2 #388

# Number of measurements per grid point
num_measurements = 200

# Number of grid points to select for training
num_train_points = int(0.75 * num_grid_points)  # 291

# Initialize empty lists for training and test sets
train_set = []
test_set = []

# For each array (representing a different environment)
for array in [CTF_class, CTF_corridor, CTF_lab, CTF_SC]:
    # Reshape the array to separate the measurements for each grid point
    reshaped_array = array.reshape(num_grid_points, num_measurements, -1, 2)
    
    # Randomly select grid points for training
    train_points = random.sample(range(num_grid_points), num_train_points)
    #print(len(train_points))
        
    # Get boolean array for test points
    test_points_bool = ~np.isin(range(num_grid_points), train_points)

    # Print test points
    # test_points = np.arange(num_grid_points)[test_points_bool]
    # print("Test points len:", len(test_points.tolist()))
    
    # Add the selected grid points to the training set and the rest to the test set
    train_set.append(reshaped_array[train_points])
    test_set.append(reshaped_array[test_points_bool])
    
    # ~ is the logical NOT operator, so ~np.isin(shuffle_index, train_points) gives all the indices not in train_points.
# Concatenate the data from all environments
train_set = np.concatenate(train_set, axis=0)
test_set = np.concatenate(test_set, axis=0)

print("Training set:",train_set.shape)
print("Testing set:",test_set.shape)

Training set: (1164, 200, 101, 2)
Testing set: (388, 200, 101, 2)


In [ ]:
# Reshaping the arrays
large_X_train = train_set.reshape([-1,101,2])
large_X_test = test_set.reshape([-1,101,2])
print("large_X_train after reshape:", large_X_train.shape)
print("large_X_test after reshape:", large_X_test.shape)

large_X_train after reshape: (232800, 101, 2)
large_X_test after reshape: (77600, 101, 2)


In [ ]:
# Create Labels for training data
#static
label1 = np.ones([291*200]) * 0  # class
label2 = np.ones([291*200]) * 1  # corridor
label3 = np.ones([291*200]) * 2  # lab
label4 = np.ones([291*200]) * 3  # SC

large_Y_train = np.concatenate([label1, label2, label3, label4])

label5 = np.ones([97*200]) * 0  # class
label6 = np.ones([97*200]) * 1  # corridor
label7 = np.ones([97*200]) * 2  # lab
label8 = np.ones([97*200]) * 3  # SC

large_Y_test = np.concatenate([label5, label6, label7, label8])

print("Training labels:", large_Y_train.shape)
print("Testing labels:", large_Y_test.shape)

Training labels: (232800,)
Testing labels: (77600,)


In [ ]:
# Shuffle Data
shuffle_index1 = random.sample(range(0,232800), 232800)
#print(shuffle_index1)

# Shuffle data using the shuffled indices
large_X_train_new = large_X_train[shuffle_index1, :, :]
print(shuffle_index1[232797:])

large_Y_train = large_Y_train[shuffle_index1]
large_Y_train

[51585, 184234, 79660]


array([0., 3., 1., ..., 0., 3., 1.])

In [ ]:
# Shuffle testing data
shuffle_index2 = random.sample(range(0,77600), 77600)

# Shuffle testing data using the shuffled indices
large_X_test_new = large_X_test[shuffle_index2, :, :]
#print(shuffle_index2[0:3])
print(shuffle_index2[77597:])

large_Y_test = large_Y_test[shuffle_index2]
large_Y_test

[17439, 76528, 62405]


array([3., 0., 3., ..., 0., 3., 3.])

In [ ]:
from keras.utils import to_categorical

# Converting labels to categorical one-hot encoding
y_train = to_categorical(large_Y_train, num_classes=4) # (232000,)
y_test = to_categorical(large_Y_test, num_classes=4) # (78400,)

X_train = large_X_train_new # (232000, 101, 2) 
X_test = large_X_test_new # (77600, 101, 2) 

In [ ]:
# Access the 4th item in each loaded variable
#static
CTF_class_static_array = CTF_class_static[list(CTF_class_static.keys())[3]].T
CTF_corridor_static_array = CTF_corridor_static[list(CTF_corridor_static.keys())[3]].T
CTF_lab_static_array = CTF_lab_static[list(CTF_lab_static.keys())[3]].T
CTF_SC_static_array = CTF_SC_static[list(CTF_SC_static.keys())[3]].T

# mov
CTF_class_mov_array = CTF_class_mov[list(CTF_class_mov.keys())[3]].T
CTF_corridor_mov_array = CTF_corridor_mov[list(CTF_corridor_mov.keys())[3]].T
CTF_lab_mov_array = CTF_lab_mov[list(CTF_lab_mov.keys())[3]].T
CTF_SC_mov_array = CTF_SC_mov[list(CTF_SC_mov.keys())[3]].T

CTF_class_static_array.shape

(38800, 2, 101)

In [ ]:
# Combine real and imaginary parts for the 4th item in each loaded variable using stack
CTF_class = np.concatenate((CTF_class_static_array, CTF_class_mov_array), axis=0)
CTF_corridor = np.concatenate((CTF_corridor_static_array, CTF_corridor_mov_array), axis=0)
CTF_lab = np.concatenate((CTF_lab_static_array, CTF_lab_mov_array), axis=0)
CTF_SC = np.concatenate((CTF_SC_static_array, CTF_SC_mov_array), axis=0)

In [ ]:
CTF_SC.shape

(77600, 2, 101)

In [ ]:
# Combine 145 training points randomly selected from each env dataset into a single training set array, 
# and combine 49 testing points randomly selected from each dataset into a single testing set array.

# Number of unique grid points
num_grid_points = 194*2 #388

# Number of measurements per grid point
num_measurements = 200

# Number of grid points to select for training
num_train_points = int(0.75 * num_grid_points)  # 291

# Initialize empty lists for training and test sets
train_set = []
test_set = []

# For each array (representing a different environment)
for array in [CTF_class, CTF_corridor, CTF_lab, CTF_SC]:
    # Reshape the array to separate the measurements for each grid point
    reshaped_array = array.reshape(num_grid_points, num_measurements, -1, 2)
    
    # Randomly select grid points for training
    train_points = random.sample(range(num_grid_points), num_train_points)
    #print(len(train_points))
        
    # Get boolean array for test points
    test_points_bool = ~np.isin(range(num_grid_points), train_points)

    # Print test points
    # test_points = np.arange(num_grid_points)[test_points_bool]
    # print("Test points len:", len(test_points.tolist()))
    
    # Add the selected grid points to the training set and the rest to the test set
    train_set.append(reshaped_array[train_points])
    test_set.append(reshaped_array[test_points_bool])
    
    # ~ is the logical NOT operator, so ~np.isin(shuffle_index, train_points) gives all the indices not in train_points.
# Concatenate the data from all environments
train_set = np.concatenate(train_set, axis=0)
test_set = np.concatenate(test_set, axis=0)

print("Training set:",train_set.shape)
print("Testing set:",test_set.shape)

Training set: (1164, 200, 101, 2)
Testing set: (388, 200, 101, 2)


In [ ]:
# Reshaping the arrays
large_X_train = train_set.reshape([-1,101,2])
large_X_test = test_set.reshape([-1,101,2])
print("large_X_train after reshape:", large_X_train.shape)
print("large_X_test after reshape:", large_X_test.shape)

large_X_train after reshape: (232800, 101, 2)
large_X_test after reshape: (77600, 101, 2)


In [ ]:
# Create Labels for training data
#static
label1 = np.ones([291*200]) * 0  # class
label2 = np.ones([291*200]) * 1  # corridor
label3 = np.ones([291*200]) * 2  # lab
label4 = np.ones([291*200]) * 3  # SC

large_Y_train = np.concatenate([label1, label2, label3, label4])

label5 = np.ones([97*200]) * 0  # class
label6 = np.ones([97*200]) * 1  # corridor
label7 = np.ones([97*200]) * 2  # lab
label8 = np.ones([97*200]) * 3  # SC

large_Y_test = np.concatenate([label5, label6, label7, label8])

print("Training labels:", large_Y_train.shape)
print("Testing labels:", large_Y_test.shape)

Training labels: (232800,)
Testing labels: (77600,)


In [ ]:
# Shuffle Data
shuffle_index1 = random.sample(range(0,232800), 232800)
#print(shuffle_index1)

# Shuffle data using the shuffled indices
large_X_train_new = large_X_train[shuffle_index1, :, :]
print(shuffle_index1[232797:])

large_Y_train = large_Y_train[shuffle_index1]
large_Y_train

[51585, 184234, 79660]


array([0., 3., 1., ..., 0., 3., 1.])

In [ ]:
# Shuffle testing data
shuffle_index2 = random.sample(range(0,77600), 77600)

# Shuffle testing data using the shuffled indices
large_X_test_new = large_X_test[shuffle_index2, :, :]
#print(shuffle_index2[0:3])
print(shuffle_index2[77597:])

large_Y_test = large_Y_test[shuffle_index2]
large_Y_test

[17439, 76528, 62405]


array([3., 0., 3., ..., 0., 3., 3.])

In [ ]:
from keras.utils import to_categorical

# Converting labels to categorical one-hot encoding
y_train = to_categorical(large_Y_train, num_classes=4) # (232000,)
y_test = to_categorical(large_Y_test, num_classes=4) # (78400,)

X_train = large_X_train_new # (232000, 101, 2) 
X_test = large_X_test_new # (77600, 101, 2) 

In [81]:
model.save("model_freq_101_relu_42.keras")

In [82]:
from tensorflow import keras
model_path1 = "model_freq_101_relu_42.keras"
model1 = keras.models.load_model(model_path1)


In [83]:
model1.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_16 (Conv2D)          (None, 99, 2, 5)          20        
                                                                 
 batch_normalization_16 (Ba  (None, 99, 2, 5)          20        
 tchNormalization)                                               
                                                                 
 re_lu_24 (ReLU)             (None, 99, 2, 5)          0         
                                                                 
 conv2d_17 (Conv2D)          (None, 97, 2, 5)          80        
                                                                 
 batch_normalization_17 (Ba  (None, 97, 2, 5)          20        
 tchNormalization)                                               
                                                                 
 re_lu_25 (ReLU)             (None, 97, 2, 5)         

### QAT

In [90]:
import tensorflow_model_optimization as tfmot

# Step 1: Annotate
annotated_model = tfmot.quantization.keras.quantize_annotate_model(model1)

# Step 2: Apply QAT
quant_aware_model = tfmot.quantization.keras.quantize_apply(annotated_model)



In [91]:
quant_aware_model.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer_4 (Quantize  (None, 101, 2, 1)         3         
 Layer)                                                          
                                                                 
 quant_conv2d_16 (QuantizeW  (None, 99, 2, 5)          31        
 rapperV2)                                                       
                                                                 
 quant_batch_normalization_  (None, 99, 2, 5)          21        
 16 (QuantizeWrapperV2)                                          
                                                                 
 quant_re_lu_24 (QuantizeWr  (None, 99, 2, 5)          3         
 apperV2)                                                        
                                                                 
 quant_conv2d_17 (QuantizeW  (None, 97, 2, 5)         

In [92]:
from tensorflow.keras.optimizers import Adam

quant_aware_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_qat = quant_aware_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=3,         
    batch_size=256
)

quant_aware_model.save('model_freq_101_relu_qat.keras')


Epoch 1/3
910/910 [==============================] - 37s 37ms/step - loss: 0.0096 - accuracy: 0.9961 - val_loss: 0.7389 - val_accuracy: 0.8654
Epoch 2/3
910/910 [==============================] - 33s 37ms/step - loss: 2.4911e-04 - accuracy: 1.0000 - val_loss: 0.0183 - val_accuracy: 0.9954
Epoch 3/3
910/910 [==============================] - 34s 37ms/step - loss: 1.1552e-04 - accuracy: 1.0000 - val_loss: 0.0230 - val_accuracy: 0.9945


In [93]:
def representative_data_gen():
    for i in range(1000):
        # the shape is (1, 101, 2, 1)
        data = X_train[i:i+1]
        if data.ndim == 3:
            data = np.expand_dims(data, axis=-1)
        yield [data.astype(np.float32)]


import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(quant_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_qat_model = converter.convert()
with open("model_freq_101_relu_qat_int8.tflite", "wb") as f:
    f.write(tflite_qat_model)


INFO:tensorflow:Assets written to: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmpmka5ajqp/assets


INFO:tensorflow:Assets written to: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmpmka5ajqp/assets
/opt/anaconda3/envs/SDR/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2025-07-05 17:45:50.818930: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-05 17:45:50.819278: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-05 17:45:50.826009: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmpmka5ajqp
2025-07-05 17:45:50.831562: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-05 17:45:50.831585: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/

In [94]:
import os
print("TFLite int8 model size (MB):", os.path.getsize("model_freq_101_relu_qat_int8.tflite") / (1024 * 1024))


TFLite int8 model size (MB): 0.1911163330078125


In [97]:
import os

# Model paths
keras_model_path = "model_freq_101_relu_42.keras"
tflite_model_path = "model_freq_101_relu_qat_int8.tflite"

# Get sizes in bytes
keras_size = os.path.getsize(keras_model_path)
tflite_size = os.path.getsize(tflite_model_path)

# Convert to MB
keras_size_mb = keras_size / (1024 * 1024)
tflite_size_mb = tflite_size / (1024 * 1024)

print(f"\nKeras model size:  {keras_size_mb:.3f} MB")
print(f"TFLite int8 size:  {tflite_size_mb:.3f} MB")

# Percentage reduction
size_reduction = (keras_size_mb - tflite_size_mb) / keras_size_mb * 100
print(f"TFLite int8 is {size_reduction:.2f}% smaller than the Keras model.")



Keras model size:  2.284 MB
TFLite int8 size:  0.191 MB
TFLite int8 is 91.63% smaller than the Keras model.


In [98]:
import tensorflow as tf
from tensorflow import keras
keras_model = keras.models.load_model("model_freq_101_relu_42.keras")


tflite_path = "model_freq_101_relu_qat_int8.tflite"
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale, input_zero_point = input_details[0]['quantization']
output_scale, output_zero_point = output_details[0]['quantization']


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [99]:
import numpy as np

# ----------- Keras Model Inference -----------
keras_preds = keras_model.predict(X_test)
keras_pred_classes = np.argmax(keras_preds, axis=1)

# ----------- TFLite Model Inference -----------
tflite_pred_classes = []
for i in range(len(X_test)):
    sample = X_test[i:i+1]
    if sample.ndim == 3:
        sample = np.expand_dims(sample, -1)  # Add channel dim
    sample_int8 = np.round(sample / input_scale + input_zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    output_int8 = interpreter.get_tensor(output_details[0]['index'])
    output = (output_int8.astype(np.float32) - output_zero_point) * output_scale
    pred_class = np.argmax(output)
    tflite_pred_classes.append(pred_class)

# ----------- Compare to True Labels -----------
y_true = np.argmax(y_test, axis=1) if y_test.ndim > 1 else y_test

keras_acc = np.mean(keras_pred_classes == y_true)
tflite_acc = np.mean(tflite_pred_classes == y_true)

print(f"Original model accuracy:   {keras_acc:.4f}")
print(f"TFLite int8 accuracy:   {tflite_acc:.4f}")


2425/2425 [==============================] - 10s 4ms/step
Original model accuracy:   0.9951
TFLite int8 accuracy:   0.9945


In [100]:
import time
import numpy as np

# ---------- Keras Model Inference Time ----------

# Time inference on the entire test set
start = time.time()
keras_preds = keras_model.predict(X_test)
keras_total_time = time.time() - start
keras_avg_time = keras_total_time / len(X_test)

print(f"\nKeras model total inference time (all test samples): {keras_total_time:.3f} s")
print(f"Keras model avg. inference time per sample: {keras_avg_time*1e3:.3f} ms")

# ---------- TFLite Model Inference Time ----------
start = time.time()
for i in range(len(X_test)):
    sample = X_test[i:i+1]
    if sample.ndim == 3:
        sample = np.expand_dims(sample, -1)  # Add channel dim
    sample_int8 = np.round(sample / input_scale + input_zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
tflite_total_time = time.time() - start
tflite_avg_time = tflite_total_time / len(X_test)

print(f"TFLite int8 model total inference time (all test samples): {tflite_total_time:.3f} s")
print(f"TFLite int8 model avg. inference time per sample: {tflite_avg_time*1e3:.3f} ms")

# % difference:
total_improvement = (keras_total_time - tflite_total_time) / keras_total_time * 100
print(f"\nTFLite int8 is {total_improvement:.2f}% faster on total test set inference time.")


2425/2425 [==============================] - 10s 4ms/step

Keras model total inference time (all test samples): 10.850 s
Keras model avg. inference time per sample: 0.140 ms
TFLite int8 model total inference time (all test samples): 2.837 s
TFLite int8 model avg. inference time per sample: 0.037 ms

TFLite int8 is 73.85% faster on total test set inference time.


### PTQ

In [101]:
import tensorflow as tf

# Path to your original Keras model (float32, NOT quantization-aware trained)
keras_model_path = "model_freq_101_relu_42.keras"

# Load the model
from tensorflow import keras
float_model = keras.models.load_model(keras_model_path)

# Representative dataset function (same shape as for QAT)
def representative_data_gen():
    for i in range(1000):
        sample = X_test[i:i+1]
        if sample.ndim == 3:
            sample = np.expand_dims(sample, -1)
        yield [sample.astype(np.float32)]


# Convert to int8 TFLite using PTQ
ptq_tflite_path = "model_freq_101_relu_ptq_int8.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(float_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

ptq_tflite_model = converter.convert()
with open(ptq_tflite_path, "wb") as f:
    f.write(ptq_tflite_model)

print("PTQ TFLite int8 model saved at:", ptq_tflite_path)


INFO:tensorflow:Assets written to: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmp9hl8_uha/assets


INFO:tensorflow:Assets written to: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmp9hl8_uha/assets
/opt/anaconda3/envs/SDR/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2025-07-05 17:49:19.781651: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


PTQ TFLite int8 model saved at: model_freq_101_relu_ptq_int8.tflite


2025-07-05 17:49:19.781682: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-05 17:49:19.782051: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmp9hl8_uha
2025-07-05 17:49:19.785686: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-05 17:49:19.785709: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmp9hl8_uha
2025-07-05 17:49:19.795962: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-05 17:49:19.899337: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/sd/hj30l0b96zj8lbk9ktvcxvpr0000gn/T/tmp9hl8_uha
2025-07-05 17:49:19.931325: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took

In [103]:
# Load PTQ TFLite model
interpreter_ptq = tf.lite.Interpreter(model_path=ptq_tflite_path)
interpreter_ptq.allocate_tensors()
input_details_ptq = interpreter_ptq.get_input_details()
output_details_ptq = interpreter_ptq.get_output_details()
input_scale_ptq, input_zero_point_ptq = input_details_ptq[0]['quantization']
output_scale_ptq, output_zero_point_ptq = output_details_ptq[0]['quantization']

# Inference on all test samples
ptq_pred_classes = []
for i in range(len(X_test)):
    sample = X_test[i:i+1]
    # Ensure shape is (1, 101, 2, 1)
    if sample.ndim == 3:
        sample = np.expand_dims(sample, -1)
    sample_int8 = np.round(sample / input_scale_ptq + input_zero_point_ptq).astype(np.int8)
    interpreter_ptq.set_tensor(input_details_ptq[0]['index'], sample_int8)
    interpreter_ptq.invoke()
    output_int8 = interpreter_ptq.get_tensor(output_details_ptq[0]['index'])
    output = (output_int8.astype(np.float32) - output_zero_point_ptq) * output_scale_ptq
    pred_class = np.argmax(output)
    ptq_pred_classes.append(pred_class)
ptq_pred_classes = np.array(ptq_pred_classes)


# True labels
y_true = np.argmax(y_test, axis=1) if y_test.ndim > 1 else y_test

# PTQ Accuracy
ptq_acc = np.mean(ptq_pred_classes == y_true)
print(f"PTQ TFLite int8 model accuracy: {ptq_acc:.4f}")


PTQ TFLite int8 model accuracy: 0.9951


In [104]:
import time

# Timing PTQ inference
start = time.time()
for i in range(len(X_test)):
    sample = X_test[i:i+1]
    if sample.ndim == 3:
        sample = np.expand_dims(sample, -1)
    sample_int8 = np.round(sample / input_scale_ptq + input_zero_point_ptq).astype(np.int8)
    interpreter_ptq.set_tensor(input_details_ptq[0]['index'], sample_int8)
    interpreter_ptq.invoke()
    _ = interpreter_ptq.get_tensor(output_details_ptq[0]['index'])
ptq_total_time = time.time() - start
ptq_avg_time = ptq_total_time / len(X_test)

print(f"PTQ TFLite int8 model total inference time: {ptq_total_time:.3f} s")
print(f"PTQ TFLite int8 model avg. inference time per sample: {ptq_avg_time*1e3:.3f} ms")


PTQ TFLite int8 model total inference time: 3.309 s
PTQ TFLite int8 model avg. inference time per sample: 0.043 ms


In [105]:
import os

ptq_size = os.path.getsize(ptq_tflite_path) / (1024 * 1024)
qat_size = os.path.getsize("model_freq_101_relu_qat_int8.tflite") / (1024 * 1024)
keras_size = os.path.getsize(keras_model_path) / (1024 * 1024)

print(f"\nPTQ TFLite int8 size: {ptq_size:.3f} MB")
print(f"QAT TFLite int8 size: {qat_size:.3f} MB")
print(f"Keras float model size: {keras_size:.3f} MB")

ptq_reduction = (keras_size - ptq_size) / keras_size * 100
print(f"PTQ model is {ptq_reduction:.2f}% smaller than Keras model.")



PTQ TFLite int8 size: 0.191 MB
QAT TFLite int8 size: 0.191 MB
Keras float model size: 2.284 MB
PTQ model is 91.65% smaller than Keras model.


In [106]:
print("\nSummary:")
print(f"Original model accuracy:       {keras_acc:.4f}")
print(f"QAT TFLite int8 accuracy:   {tflite_acc:.4f}")
print(f"PTQ TFLite int8 accuracy:   {ptq_acc:.4f}")
print(f"Keras model size:           {keras_size:.3f} MB")
print(f"QAT TFLite int8 size:       {qat_size:.3f} MB")
print(f"PTQ TFLite int8 size:       {ptq_size:.3f} MB")



Summary:
Original model accuracy:       0.9951
QAT TFLite int8 accuracy:   0.9945
PTQ TFLite int8 accuracy:   0.9951
Keras model size:           2.284 MB
QAT TFLite int8 size:       0.191 MB
PTQ TFLite int8 size:       0.191 MB
